In [0]:
from pyspark.sql.functions import lit 


##CPT Codes

In [0]:
df_cptcodes = spark.read.option('header', True).csv(path = 'abfss://landing@adlshosptial.dfs.core.windows.net/CPTcodes/cptcodes.csv')

for col in df_cptcodes.columns:
    new_col= col.replace(' ','_').lower()
    df_cptcodes= df_cptcodes.withColumnRenamed(col, new_col)

df_cptcodes.write.mode('overwrite').parquet(path= 'abfss://bronze@adlshosptial.dfs.core.windows.net/CPTcodes')

##Claims Data

Claims - Hospital A

In [0]:
df_claims_hospA= spark.read.option('header', True).csv(path= 'abfss://landing@adlshosptial.dfs.core.windows.net/Claims/HospitalA_claim_data.csv')

df_claims_hospA= df_claims_hospA.withColumn('datasource', lit('hospA'))

df_claims_hospA.write.mode('overwrite').parquet(path = 'abfss://bronze@adlshosptial.dfs.core.windows.net/Claims/HospitalA')

Claims - Hospital B

In [0]:
df_claims_hospA= spark.read.option('header', True).csv(path= 'abfss://landing@adlshosptial.dfs.core.windows.net/Claims/HospitalB_claim_data.csv')

df_claims_hospA= df_claims_hospA.withColumn('datasource', lit('hospA'))

df_claims_hospA.write.mode('overwrite').parquet(path = 'abfss://bronze@adlshosptial.dfs.core.windows.net/Claims/HospitalB')

##ICD Codes

In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_date, lit
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DateType, BooleanType



client_id = 'c1b31e75-de89-4e9e-9738-87a4779e98e9_b7b0ac50-33a3-4ba5-abc2-4f4b1050ab15'
client_secret = 'UtTb8EIodKWT2g3BoIkgGsNbMcv0ClZKJbEij4ySzTs='
base_url = 'https://id.who.int/icd/'
current_date=datetime.now().date()

auth_url = 'https://icdaccessmanagement.who.int/connect/token'
auth_response = requests.post(auth_url, data={
    'client_id': client_id,
    'client_secret': client_secret,
    'grant_type': 'client_credentials'
})

if auth_response.status_code == 200:
    access_token = auth_response.json().get('access_token')
else:
    raise Exception(f"Failed to obtain access token: {auth_response.status_code} - {auth_response.text}")

headers = {
    'Authorization': f'Bearer {access_token}',
    'API-Version': 'v2',  # Add the API-Version header
    'Accept-Language': 'en',
}

def fetch_icd_codes(url):
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to fetch data: {response.status_code} - {response.text}")

def extract_codes(url):
    data = fetch_icd_codes(url)
    codes = []
    if 'child' in data:
        for child_url in data['child']:
            codes.extend(extract_codes(child_url))
    else:
        if 'code' in data and 'title' in data:
            # print(data['code'],data['title']['@value'])
            codes.append({
                'icd_code': data['code'],
                'icd_code_type': 'ICD-10',
                'code_description': data['title']['@value'],
                'inserted_date': current_date,
                'updated_date': current_date,
                'is_current_flag': True
            })
    return codes

# Start from the root URL
root_url = 'https://id.who.int/icd/release/10/2019/A00-A09'
icd_codes = extract_codes(root_url)


# Define the schema explicitly
schema = StructType([
    StructField("icd_code", StringType(), True),
    StructField("icd_code_type", StringType(), True),
    StructField("code_description", StringType(), True),
    StructField("inserted_date", DateType(), True),
    StructField("updated_date", DateType(), True),
    StructField("is_current_flag", BooleanType(), True)
])

# Create a DataFrame using the defined schema
print(icd_codes)
df = spark.createDataFrame(icd_codes, schema=schema)
df.write.format("parquet").mode("append").save('abfss://bronze@adlshosptial.dfs.core.windows.net/ICD_Codes')


##NPI Extract

In [0]:
from datetime import date
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DateType

# In Databricks notebooks, spark is already available, but initializing is harmless
spark = SparkSession.builder.appName("NPI Data").getOrCreate()

# Current date as Python date object
current_date = date.today()

# Base URL for the NPI Registry API
base_url = "https://npiregistry.cms.hhs.gov/api/"

# Parameters for initial API request
params = {
    "version": "2.1",
    "state": "CA",
    "city": "Los Angeles",
    "limit": 20,
}

# Make the initial API request
response = requests.get(base_url, params=params)

if response.status_code == 200:
    npi_data = response.json()
    npi_list = [result["number"] for result in npi_data.get("results", [])]

    detailed_results = []

    for npi in npi_list:
        detail_params = {"version": "2.1", "number": npi}
        detail_response = requests.get(base_url, params=detail_params)

        if detail_response.status_code == 200:
            detail_data = detail_response.json()
            if "results" in detail_data and detail_data["results"]:
                for result in detail_data["results"]:
                    npi_number = result.get("number")
                    basic_info = result.get("basic", {})
                    if result.get("enumeration_type") == "NPI-1":
                        fname = basic_info.get("first_name", "")
                        lname = basic_info.get("last_name", "")
                    else:
                        fname = basic_info.get("authorized_official_first_name", "")
                        lname = basic_info.get("authorized_official_last_name", "")
                    position = basic_info.get("authorized_official_title_or_position", "")
                    organisation = basic_info.get("organization_name", "")
                    last_updated = basic_info.get("last_updated", "")

                    detailed_results.append({
                        "npi_id": npi_number,
                        "first_name": fname,
                        "last_name": lname,
                        "position": position,
                        "organisation_name": organisation,
                        "last_updated": last_updated,
                        "refreshed_at": current_date   # Python date object
                    })

    if detailed_results:
        # Explicit schema with DateType for refreshed_at
        schema = StructType([
            StructField("npi_id", StringType(), True),
            StructField("first_name", StringType(), True),
            StructField("last_name", StringType(), True),
            StructField("position", StringType(), True),
            StructField("organisation_name", StringType(), True),
            StructField("last_updated", StringType(), True),
            StructField("refreshed_at", DateType(), True)
        ])

        df = spark.createDataFrame(detailed_results, schema=schema)
        display(df)

        # Save to ADLS (correct your storage account/container name)
        df.write.format("parquet").mode("overwrite").save(
            "abfss://bronze@adlshosptial.dfs.core.windows.net/NPI_Extract"
        )

        # Save as Delta table
        df.write.format("delta").mode("overwrite").saveAsTable("npi_extract")

    else:
        print("No detailed results found.")
else:
    print(f"Failed to fetch data: {response.status_code} - {response.text}")
